# DAPPC - LAB 1 (2025 - 2026)
## Dataset preparation (cleaning + comorbidity encoding)

In this notebook you will:

1. Load the dataset  
2. Inspect initial size and print a table with **(column index, column name, dtype)**  
3. Analyze missing values **per variable** and drop variables above a threshold  
4. Analyze missing values **per subject (row)** and drop subjects above a threshold  
5. Encode comorbidities into 6 aggregated variables  
6. Save the final prepared dataset for the next labs

**Outcome (3 classes):**
- `0` = extubation failure  
- `1` = extubation success  
- `2` = death during ICU stay


---

## 0) Setup
Run this cell first.


In [ ]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook


---

## 1) Load the dataset

---

## 2) Initial exploration: size + column index table

We check:
- number of rows and columns
- a **table** with: column index, column name, dtype

This is useful because some steps (e.g., comorbidity encoding) use **column indices**.


In [ ]:
# === TODO: set your file path and sheet name ===
file_path = "/content/Dataset_DAPPC_2026.xlsx"
sheet_name = "Dataset"
df = pd.read_excel(file_path, sheet_name=sheet_name)
df.head()

,subject_id,hadm_id,stay_id,ICU,ICU_intime,ICU_outtime,gender,age,height,weight,...,first_PEEP,std_PEEP,first_PIP,std_PIP,first_mean_airway_pressure,std_mean_airway_pressure,first_plateau_pressure,std_plateau_pressure,vent_duration_hours,outcome
0,13710366,26676995,36043579,1,2172-04-05 04:00:00,2172-04-09 08:17:22,1,66,178.0,113.3,...,5.0,310.799588,26.0,5.971675,13.0,1.923269,19.0,1.909727,96.0,2
1,12970079,23949170,36004625,1,2164-12-22 19:48:19,2165-01-03 21:30:54,1,53,NaN,106.7,...,5.0,284.163626,18.0,11.940840,12.0,6.393488,10.0,8.337289,230.0,1
2,17682100,22097504,34901461,1,2173-08-23 01:12:00,2173-08-27 21:28:52,0,66,NaN,46.1,...,460.0,210.569244,20.0,5.387228,8.0,2.184796,17.0,1.224745,34.0,1
3,15545849,24922530,39025631,3,2172-08-02 16:32:00,2172-09-09 12:44:11,1,49,183.0,106.0,...,5.0,1042.283061,12.0,7.166729,7.0,4.233572,11.0,4.877563,399.0,2
4,10288279,21750307,34078845,1,2114-09-07 20:19:42,2114-09-15 13:24:09,0,77,NaN,49.3,...,8.0,185.553721,21.0,4.283036,13.0,1.405487,17.0,3.239768,155.0,2


In [ ]:
print("Initial shapes of the dataset (rows, cols):", df.shape)

col_table = pd.DataFrame({
    "col_index": np.arange(df.shape[1]),
    "col_name": df.columns.astype(str),
    "dtype": df.dtypes.astype(str)
})

#If you want to display all the columns, de-comment the following instruction
# pd.set_option("display.max_rows", None)

display(col_table)


Initial shapes of the dataset (rows, cols): (4000, 120)


,col_index,col_name,dtype
subject_id,0,subject_id,int64
hadm_id,1,hadm_id,int64
stay_id,2,stay_id,int64
ICU,3,ICU,int64
ICU_intime,4,ICU_intime,object
...,...,...,...
std_mean_airway_pressure,115,std_mean_airway_pressure,float64
first_plateau_pressure,116,first_plateau_pressure,float64
std_plateau_pressure,117,std_plateau_pressure,float64
vent_duration_hours,118,vent_duration_hours,float64


---

## 3) Missing values per variable (column) + drop with threshold

We compute missing values **per column** and optionally drop columns with too many missing values.
We want to drop the features with at lest

In [ ]:
# STEP 1 — Remove columns with at least 5% missing values

# Set the threshold
col_missing_threshold = 0.05

# Compute missing count and missing percentage for each column
missing_count = df.isna().sum()
missing_perc = missing_count / df.shape[0]

#  Identify columns to drop (missing % > threshold)
cols_to_drop = df.columns[missing_perc > col_missing_threshold]

print(f"Columns to drop (>{col_missing_threshold*100:.0f}% missing):", len(cols_to_drop))

# 5) Create df_1 by dropping those columns
df_1 = df.drop(columns=cols_to_drop)

print("Initial shapes of the dataset (rows, cols):", df.shape)
print("Obtained shapes of the dataset (rows, cols):", df_1.shape)


Columns to drop (>5% missing): 5
Initial shapes of the dataset (rows, cols): (4000, 120)
Obtained shapes of the dataset (rows, cols): (4000, 115)


---

## 4) Missing values per subject (row) + drop with threshold

Now we analyze missing values **per row** (per patient/subject).
We want to drop the subjects with at least 5% of MVs.

> Note: You may want to compute missing rate only on **feature columns** (not IDs/outcome). If needed, adapt the code.


In [ ]:
# STEP 2 — Remove rows (patients) with at least 5% missing values

# Set the threshold
row_missing_threshold = 0.05

# Compute missing percentage for each row in df_1
row_missing_perc = df_1.isna().sum(axis=1) / df_1.shape[1]

# Identify rows to drop (missing % > threshold)
rows_to_drop = row_missing_perc[row_missing_perc > row_missing_threshold].index.tolist()

print(f"Rows to drop (>{row_missing_threshold*100:.0f}% missing):", len(rows_to_drop))

# Create df_2 by dropping those rows and resetting the index
df_2 = df_1.drop(index=rows_to_drop).reset_index(drop=True)

print("Initial shapes of the dataset (rows, cols):", df_1.shape)
print("Obtained shapes of the dataset (rows, cols):", df_2.shape)

Rows to drop (>5% missing): 107
Initial shapes of the dataset (rows, cols): (4000, 115)
Obtained shapes of the dataset (rows, cols): (3893, 115)


---

## 5) Encode comorbidities into categories

We aggregate the original comorbidity columns (0/1) into:
- `comorb_total`: total number of comorbidities
- `comorb_cat1` … `comorb_cat5`: 1 if **at least one** comorbidity is present in that category

The five categories are:
1. Cardiac and cardiovascular diseases
2. Respiratory and pulmonary diseases
3. Metabolic, endocrine and renal diseases
4. Neurological, neuromuscolar and phychiatric disorders
5. System, immunologic and oncologic conditions.

Finally, we **drop** the original comorbidity columns.

⚠️ Update indices if the final column order changes.


In [ ]:
# Show the index and the header of all the columns to identify the ones related to Comorbidities
col_table = pd.DataFrame({
    "col_index": np.arange(df_2.shape[1]),
    "col_name": df_2.columns.astype(str),
    "dtype": df_2.dtypes.astype(str)
})

pd.set_option("display.max_rows", None)
display(col_table)

# - Identify the first and last comorbidity column indices from the table above
idx_start = 9 # 'adrenal_disorders'
idx_end = 54 # Exclusive: 'valvular_disease' is at index 53


,col_index,col_name,dtype
subject_id,0,subject_id,int64
hadm_id,1,hadm_id,int64
stay_id,2,stay_id,int64
ICU,3,ICU,int64
ICU_intime,4,ICU_intime,object
ICU_outtime,5,ICU_outtime,object
gender,6,gender,int64
age,7,age,int64
weight,8,weight,float64
adrenal_disorders,9,adrenal_disorders,int64


In [ ]:
# ================== COMORBIDITY GROUPS ==================
# 0 = absent, 1 = present
# Check the names before running the code: typos can be present!

comorbidity_categories = {
    1: ["cardiac_arrhythmia","congestive_heart_failure","coronary_artery_disease","hypertension","myocardial_infarction","peripheral_vascular_disease","valvular_disease"],
    2: ["asthma","chronic_pulmonary_disease","interstitial_lung_disease","pulmonary_circulatory_disease","pulmonary_hypertension","sleep_apnea_OSAS"],
    3: ["adrenal_disorders","chronic_kidney_disease","cirrhosis","diabetes_with_complications","diabetes_without_complications","fluid_electrolyte_disorders","mild_liver_disease","severe_liver_disease","malnutrition","obesity","peptic_ulcer_disease","renal_failure","thyroid_disorders"],
    4: ["alcohol_abuse","central_nervous_system_lymphoma","cerebrovascular_disease","dementia","depression","drug_abuse","paraplegia","neuromuscular_disease","peripheral_neuropathy","seizure","smoke","stroke"],
    5: ["aids","blood_loss_anemia","deficiency_anemia","immunosuppression","malignant_tumor","metastatic_solid_tumor","rheumatic_disease"]
}

cat_name = {
    1: "cardiac_cardiovascular",
    2: "respiratory_pulmonary",
    3: "metabolic_endocrine_renal",
    4: "neurological_neuromuscular_psychiatric",
    5: "systemic_immune_oncologic"
}

# ================== CONSISTENCY CHECK ==================
missing_vars = {}

for cat, vars_list in comorbidity_categories.items():
    missing = [v for v in vars_list if v not in df_2.columns]
    if missing:
        missing_vars[cat] = missing

if not missing_vars:
    print("All comorbidity variables were found in the dataset.")
else:
    print("Some comorbidity variables were NOT found in the dataset:")
    for cat, vars_list in missing_vars.items():
        print(f"  Category {cat}: {vars_list}")


All comorbidity variables were found in the dataset.


In [ ]:
# STEP 4 — Comorbidity encoding (aggregation into groups)
# Goal: Create 5 new columns (0/1):
# - 1 if at least one comorbidity in that group is 1
# - 0 otherwise
df_3 = df_2.copy()

# Extract comorbidity columns from df_2 using idx_start and idx_end
comorbidity_cols = df_2.columns[idx_start:idx_end]

# Calculate comorb_total
df_3['comorb_total'] = df_2[comorbidity_cols].sum(axis=1)

# Identify the absence/presence of each category
for cat_id, vars_list in comorbidity_categories.items():
    # Filter for comorbidity variables that are actually present in df_2
    cat_cols_present = [col for col in vars_list if col in df_2.columns]
    if cat_cols_present: # Only add column if there are comorbidities in this category
        df_3[f'comorb_cat{cat_name[cat_id]}'] = (df_2[cat_cols_present].sum(axis=1) > 0).astype(int)

# Remove the original comorbidity columns
df_3 = df_3.drop(columns=comorbidity_cols)

print("Dataset shape after comorbidity grouping:", df_3.shape)

Dataset shape after comorbidity grouping: (3893, 76)


---

## 6) Final saving

Save the prepared dataset for the next steps/labs.

Choose one of the formats below.


In [ ]:
# Save the imputed dataset (kNN by class) into a new Excel sheet
new_sheet_name = "Dataset_v1"

with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df_final_by_class.to_excel(writer, sheet_name=new_sheet_name, index=False)

print(f"Dataset saved to new sheet: '{new_sheet_name}' in '{file_path}'")